Establish an environment


In [1]:
try:
  import google.colab

  IN_COLAB = True
except ImportError:
  IN_COLAB = False

if IN_COLAB:
  print("Running on Google Colab")
else:
  print("Running locally")

Running locally


In [2]:
!nvidia-smi

Sat Jan 31 11:10:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.95.05              Driver Version: 580.95.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4070        Off |   00000000:01:00.0  On |                  N/A |
|  0%   41C    P8             18W /  200W |    1675MiB /  12282MiB |      5%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----


Download data from roboflow API (API_KEY required) or from local machine


In [3]:
import os
from pathlib import Path

# Toggle between Roboflow API download or local pre-downloaded dataset
USE_ROBOFLOW_API = True

# Dataset: package_sorting
DATASET_NAME = "package_sorting"

ROBOFLOW_PROJECTS = {
  "package_sorting": {
    "workspace": "lukmik-e4qay",
    "project": "mailers-and-boxes-v10",
    "version": 6,
  },
}
FORMAT = "coco"

if USE_ROBOFLOW_API:
  if IN_COLAB:
    %pip install roboflow
    from google.colab import userdata

    ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")
    DATASET_LOCATION = f"datasets/{FORMAT}"
  else:
    from dotenv import load_dotenv

    load_dotenv()
    ROBOFLOW_API_KEY = os.getenv("ROBOFLOW_API_KEY")
    DATASET_LOCATION = f"../datasets/{FORMAT}"

  from roboflow import Roboflow

  rf = Roboflow(api_key=ROBOFLOW_API_KEY)
  proj_config = ROBOFLOW_PROJECTS[DATASET_NAME]
  project = rf.workspace(proj_config["workspace"]).project(
    proj_config["project"]
  )
  version = project.version(proj_config["version"])
  dataset = version.download(FORMAT, location=DATASET_LOCATION)
  DATASET_PATH = Path(dataset.location)
else:
  if IN_COLAB:
    DATASET_PATH = Path(f"datasets/{DATASET_NAME}/yolo_format")
  else:
    DATASET_PATH = Path(f"../datasets/{DATASET_NAME}/yolo_format")

print(f"Dataset path: {DATASET_PATH.resolve()}")
print(f"Dataset exists: {DATASET_PATH.exists()}")

loading Roboflow workspace...
loading Roboflow project...
Dataset path: /home/lukmik/Desktop/university/masters2.1/wybrane_metody_uczenia_maszynowego/projekt/datasets/package_sorting/coco
Dataset exists: True


Install dependncies

In [4]:
if IN_COLAB:
  # numpy<2.4 required due to rfdetr issue: https://github.com/apple/coremltools/issues/2633
  %pip install rfdetr 'numpy>=2.0,<2.4'

Training

In [ ]:
EPOCHS = 2

In [ ]:
from pathlib import Path
import os
import shutil

# Setup shared weights directory
if IN_COLAB:
  WEIGHTS_DIR = Path('weights')
else:
  WEIGHTS_DIR = (Path.cwd().parent / 'weights').resolve()
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

# RF-DETR will download weights to current directory by default
# We'll move them to the shared weights directory after download
RF_DETR_WEIGHTS = 'rf-detr-small.pth'
RF_DETR_WEIGHTS_PATH = WEIGHTS_DIR / RF_DETR_WEIGHTS

# Check if weights exist in shared directory
if RF_DETR_WEIGHTS_PATH.exists():
  # Copy weights to current directory for RF-DETR to use
  if not os.path.exists(RF_DETR_WEIGHTS):
    shutil.copy(RF_DETR_WEIGHTS_PATH, RF_DETR_WEIGHTS)
    print(f'Copied existing weights from {RF_DETR_WEIGHTS_PATH}')
  else:
    print(f'Using existing weights: {RF_DETR_WEIGHTS_PATH}')

from rfdetr import RFDETRSmall

OUTPUT_DIR = Path("../training_outputs/rf_detr/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model = RFDETRSmall()

model.train(
  dataset_dir=str(DATASET_PATH),
  epochs=EPOCHS,
  batch_size=8,
  grad_accum_steps=2,
  early_stopping=True,
  early_stopping_patience=5,
  output_dir=str(OUTPUT_DIR),
)

# After training, move downloaded weights to shared directory if they don't exist there
if os.path.exists(RF_DETR_WEIGHTS) and not RF_DETR_WEIGHTS_PATH.exists():
  shutil.move(RF_DETR_WEIGHTS, RF_DETR_WEIGHTS_PATH)
  print(f'Moved weights to shared directory: {RF_DETR_WEIGHTS_PATH}')